# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maaz89/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
The Content Refresh Optimization lane is fundamentally configured as a Binary Classification task (predicting a 0 or 1), which can also be interpreted as a Scoring task where the model outputs a probability between 0.0 and 1.0. The goal is to separate stable pages from decaying pages so they can be tiered by priority.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The Target Variable: is_declining_label (Binary: 1 for declining performance, 0 for stable/growing performance).

The Proxy Logic: Because "decay" isn't a single instant metric, our target acts as a proxy capturing an operational state. It flags a page if its organic performance (trend_direction) exhibits a consistent, rolling downward slope over a 90-day assessment window relative to its historical baseline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric for this task is Precision at K (specifically Precision@50 or Precision@100), supplemented by the Area Under the Precision-Recall Curve (PR-AUC).

Reasoning: Content operations teams have fixed weekly capacities (e.g., they can only rewrite 50 pages a week). If our model flags 50 pages for optimization, we need as many of those 50 as possible to be true, burning traffic-bleed cases. High Recall is secondary; maximizing the accuracy of our top recommendations avoids wasting limited creative budget on false alarms.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:

import pandas as pd
import numpy as np
import os


possible_paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    '/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv'
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break


if data_path:
    df = pd.read_csv(data_path)


    if 'is_declining_label' not in df.columns and 'trend_direction' in df.columns:
        df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
    elif 'is_declining_label' not in df.columns:

        df['is_declining_label'] = np.random.choice([0, 1], size=len(df), p=[0.76, 0.24])


    sample_cols = [col for col in ['url_hash', 'client_id', 'avg_position', 'clicks_90d', 'impressions_90d', 'is_declining_label'] if col in df.columns]
    unit_matrix = df[sample_cols].head(5)

    print("--- UNIT OF ANALYSIS DATAFRAME REPRESENTATION ---")
    print("Definition: One row = One unique URL page asset for a specific client domain.\n")
    print(unit_matrix.to_string(index=False))

else:

    print("--- UNIT OF ANALYSIS DATAFRAME REPRESENTATION (SIMULATED) ---")
    print("Definition: One row = One unique URL page asset for a specific client domain.\n")
    data = {
        'url_hash': ['url_x982f', 'url_a112b', 'url_c443z', 'url_m992d'],
        'avg_position': [1.4, 12.8, 4.2, 22.1],
        'clicks_90d': [4500, 120, 890, 12],
        'engagement_rate': [0.72, 0.41, 0.65, 0.33],
        'is_declining_label (TARGET)': [0, 1, 0, 1]
    }
    print(pd.DataFrame(data).to_string(index=False))

--- UNIT OF ANALYSIS DATAFRAME REPRESENTATION (SIMULATED) ---
Definition: One row = One unique URL page asset for a specific client domain.

 url_hash  avg_position  clicks_90d  engagement_rate  is_declining_label (TARGET)
url_x982f           1.4        4500             0.72                            0
url_a112b          12.8         120             0.41                            1
url_c443z           4.2         890             0.65                            0
url_m992d          22.1          12             0.33                            1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A naive approach to content decay would use a hardcoded rule like:Pythonif clicks_drop > 0.20 and avg_position > 5:
    return "Flag for Refresh"
While simple, this rule falls apart immediately in production because search data is highly non-linear, correlated, and noisy. Three critical patterns make this problem too messy for traditional if/else logic:Context-Dependent Scaling (The Volume Problem):A $20\%$ drop in clicks means something completely different for a high-volume head keyword page (losing 10,000 visitors) than it does for a niche long-tail page (losing 4 visitors). An if-statement applies a flat blanket rule, whereas an ML model scales its variance weights based on absolute scale and exposure (impressions_90d).Multi-Variable Interaction (The Hidden Correlation):Features do not move in isolation. A drop in traffic might look bad, but if avg_position remains steady at #1, it's just a seasonal market dip—rewriting the page won't help. Conversely, if traffic is flat but engagement_rate is plummeting while avg_position slips from #2 to #4, it signifies a critical algorithmic warning sign. Writing an if-statement that accurately maps the cross-multiplication of 5+ moving metrics across thousands of pages creates an unmaintainable, brittle web of hardcoded thresholds.The "Moving Cliff" (Non-Linearity):As discovered in Week 1, CTR decays exponentially, not linearly. The traffic impact of dropping from position 1 to position 3 is catastrophic, while moving from position 11 to position 13 is practically invisible. A decision tree or probabilistic model natively maps these step-function changes and shifting boundaries across different client profiles without manual recalibration.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.